# grad-tracking-global-toggle — ex1: no_grad context manager built on a module-level toggle

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `grad-tracking-global-toggle`. Running the final beacon cell reports progress against the `Backprop: Grad-tracking toggle` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """A minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries an optional `.recipe`
    populated by wrap_forward_fn. `requires_grad` is set by the wrapper."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Backprop: Grad-tracking toggle` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`grad-tracking-global-toggle`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "grad-tracking-global-toggle"
DD_SUBTOPIC = "Backprop: Grad-tracking toggle"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Grad-tracking global toggle — quick refresher

A module-level boolean — `grad_tracking_enabled` — gates ALL Recipe construction. When it's `False`, every `wrap_forward_fn` short-circuits to producing a Tensor with `requires_grad=False` and no Recipe attached:

```python
grad_tracking_enabled = True

def wrap_forward_fn(fwd_fn):
    def tensor_func(*args, **kwargs):
        ...
        requires_grad = grad_tracking_enabled and any(
            isinstance(a, Tensor) and a.requires_grad for a in args
        )
        ...
```

This is the analogue of PyTorch's `torch.no_grad()`. Use cases:
- **Inference** — skip graph building for speed/memory.
- **Parameter init / EMA updates** — operations that should never produce   gradients even though their inputs have `requires_grad=True`.

A context manager that flips the global to `False` on enter and restores the previous value on exit is the standard wrapper around the toggle.

### Exercise 1 — no_grad context manager built on a module-level toggle

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Apply the global-toggle + context-manager pattern: a module-level bool gates grad tracking, and a NoGrad ctx flips it off on enter and restores the previous value on exit (nesting-safe).
> Keywords: no_grad, context-manager, global-toggle, inference
> ```

**KCs targeted:** `grad-tracking-global-toggle`, `requires-grad-propagation`

We've given you a module-level `grad_tracking_enabled = True` and a `Tensor` wrapper. Implement TWO pieces:

**1. `compute_requires_grad(args)`** — return `True` iff `grad_tracking_enabled` is True AND at least one input is a Tensor with `requires_grad=True`:
```
grad_tracking_enabled and any(
    isinstance(a, Tensor) and a.requires_grad for a in args
)
```
Read the toggle FROM THE MODULE — don't snapshot it into a closure at import time. Tip: reference it via `globals()['grad_tracking_enabled']` or via `import sys; sys.modules[__name__].grad_tracking_enabled` so the latest value wins.

**2. `NoGrad` context manager** — flips the global to False on `__enter__`, restores the **previous** value on `__exit__`. Use the previous value, not a hardcoded `True`, so nested `NoGrad()` blocks behave correctly (inner exit doesn't accidentally re-enable when an outer `NoGrad` is still active).

Once both are wired, the test runs through 5 scenarios — single tensor inputs, mixed Tensor+scalar, ctx mgr toggling, nested ctx mgrs, and exception safety (toggle must restore even if the body raises).

Do NOT call `torch.autograd` or use `torch.no_grad()` — we're reimplementing them.

In [ ]:
def _get_flag():
    # globals() inside a function returns the defining module's globals
    # — works in both Colab kernels and isolated exec namespaces.
    return globals()['grad_tracking_enabled']


def _set_flag(v: bool):
    globals()['grad_tracking_enabled'] = v


def compute_requires_grad(args) -> bool:
    return _get_flag() and any(
        isinstance(a, Tensor) and a.requires_grad for a in args
    )


class NoGrad:
    def __enter__(self):
        self._prev = _get_flag()   # snapshot whatever it WAS
        _set_flag(False)
        return self

    def __exit__(self, exc_type, exc_val, exc_tb):
        _set_flag(self._prev)      # restore PREVIOUS value, not True
        return False               # don't swallow exceptions


<details><summary>Solution</summary>

```python
def _get_flag():
    # globals() inside a function returns the defining module's globals
    # — works in both Colab kernels and isolated exec namespaces.
    return globals()['grad_tracking_enabled']


def _set_flag(v: bool):
    globals()['grad_tracking_enabled'] = v


def compute_requires_grad(args) -> bool:
    return _get_flag() and any(
        isinstance(a, Tensor) and a.requires_grad for a in args
    )


class NoGrad:
    def __enter__(self):
        self._prev = _get_flag()   # snapshot whatever it WAS
        _set_flag(False)
        return self

    def __exit__(self, exc_type, exc_val, exc_tb):
        _set_flag(self._prev)      # restore PREVIOUS value, not True
        return False               # don't swallow exceptions
```

**Why read the global through `sys.modules[__name__]`.** A naive `def compute_requires_grad(args): return grad_tracking_enabled and ...` works in a script but breaks in a notebook cell that re-executes — the closure can stale-bind to the OLD value. Reading through `globals()` or `sys.modules[__name__]` always sees the current binding.

**Why restore the PREVIOUS value, not `True`.** Hardcoding `True` on exit means nested `NoGrad()` blocks corrupt each other: the inner exit re-enables grad tracking even though the outer block is still in scope. The user's nesting-test catches this directly.

**`return False` from `__exit__`.** Returning a truthy value would suppress the exception. Returning `False` (or just letting it fall off the end) lets exceptions propagate naturally — which is what we want for `try/except` to keep working through `NoGrad` blocks.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()